## Midterm Project 2

source of the Standford Amazon review data :
https://snap.stanford.edu/data/web-Amazon-links.html


### **Package Import**

In [1]:
import gdown
import gzip

### **downloading the file**

In [2]:
# Upgrade and install gdown
#!pip install --upgrade gdown

# Define file ID and output filename
file_id = "1d7T-cr17JzDeR0QxH7TGxUC3Ixc2Cp8p"
output_filename = "all.txt.gz"

# Download the file from Google Drive using gdown
# --fuzzy handles redirection/warning pages more effectively
!gdown --id {file_id} --fuzzy -O {output_filename}

# Check the first few bytes to confirm it's a valid gzip (should see something like b'\x1f\x8b')
print("\nFirst few bytes of the downloaded file:")
!head -c 4 {output_filename} | xxd


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1d7T-cr17JzDeR0QxH7TGxUC3Ixc2Cp8p
From (redirected): https://drive.google.com/uc?id=1d7T-cr17JzDeR0QxH7TGxUC3Ixc2Cp8p&confirm=t&uuid=f915687f-dc76-4dbd-8cb5-e170363acac4
To: /content/all.txt.gz
100% 11.7G/11.7G [02:29<00:00, 78.4MB/s]

First few bytes of the downloaded file:
00000000: 1f8b 0808                                ....


In [3]:
#!head -c 20 all.txt.gz


In [4]:
import gzip

def extract_single_review(filename):
    """
    We are using this function to show the structure of one review in the downloaded txt file.
    """
    review = []
    empty_count = 0

    with gzip.open(filename, 'rt', encoding='utf-8') as f:
        for line in f:
            # Check if the line is empty (after stripping whitespace)
            if line.strip() == '':
                empty_count += 1
                # Once a review has been collected, consider it finished
                if review:
                    break
            else:
                review.append(line.rstrip())
                # Reset the empty line counter when a non-empty line is encountered
                if empty_count:
                    empty_count = 0
    return review, empty_count

# File to process
filename = "all.txt.gz"

# Extract the first review
review, blank_lines = extract_single_review(filename)

# Extract and display only the categories (keys) present in the review
categories = set()
for line in review:
    if ":" in line:
        key, _ = line.split(":", 1)
        categories.add(key.strip())

print("Categories present in 1 review:")
print("----------------------------------")
for category in sorted(categories):
    print(f"- {category}")


Categories present in 1 review:
----------------------------------
- product/price
- product/productId
- product/title
- review/helpfulness
- review/profileName
- review/score
- review/summary
- review/text
- review/time
- review/userId


## **Generating smaller versions of the txt file**

In this section, we processed the large Amazon Review txt file in streaming mode to extract complete review blocks and then saved them into separate CSV files. We generated different dataset versions—1MB, 10MB, 100MB, and 1GB—ensuring that each file contains whole reviews and maintains the original data distribution.

The smaller CSV files are used for efficient training of our neural network (BERT-base) on manageable data subsets.

The complete conversion process takes approximately 1 minute and 30 seconds.

In [5]:
import gzip
import csv
import os

def stream_reviews(filename):
    """
    Generator that yields one review (as a dict) at a time.
    Each review block is separated by one or more blank lines.
    Each line in a block is formatted as "key: value".
    """
    current_block = []
    with gzip.open(filename, 'rt', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()  # Remove trailing whitespace
            if line == "":        # Blank line indicates end of block
                if current_block:
                    review_dict = {}
                    for item in current_block:
                        if ":" in item:
                            key, value = item.split(":", 1)
                            review_dict[key.strip()] = value.strip()
                    yield review_dict
                    current_block = []
            else:
                current_block.append(line)
        # Process last block if file doesn't end with a blank line
        if current_block:
            review_dict = {}
            for item in current_block:
                if ":" in item:
                    key, value = item.split(":", 1)
                    review_dict[key.strip()] = value.strip()
            yield review_dict

# Define target sizes in bytes
target_sizes = {
    "1MB": 1 * 1024 * 1024,
    "10MB": 10 * 1024 * 1024,
    "100MB": 100 * 1024 * 1024,
    "1GB": 1 * 1024 * 1024 * 1024
}

def write_dataset_version(gen_reviews, output_name, target_size):
    """
    Write reviews to a CSV file until the target file size is reached.

    Parameters:
      gen_reviews: Generator of reviews (dictionaries)
      output_name: Name of the output CSV file
      target_size: Target file size in bytes
    """
    with open(output_name, "w", newline="", encoding="utf-8") as csvfile:
        writer = None
        for review in gen_reviews:
            if writer is None:
                # Create CSV header using keys from the first review
                fieldnames = list(review.keys())
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
            writer.writerow(review)
            csvfile.flush()  # Ensure data is written to disk
            if os.path.getsize(output_name) >= target_size:
                break

# Compressed file to process
gz_filename = "all.txt.gz"

# Create the reviews generator
reviews_generator = stream_reviews(gz_filename)

# Dictionary to store the names of the generated CSV files
output_files = {}

# Write out each dataset version with simpler file names
for label, size in target_sizes.items():
    output_file = f"reviews_{label}.csv"
    output_files[label] = output_file
    print(f"Creating {label} version -> {output_file} (target size: {size} bytes)")
    write_dataset_version(reviews_generator, output_file, size)
    file_size = os.path.getsize(output_file)
    print(f"File {output_file} created, size = {file_size} bytes\n")

# Display the names of the generated CSV files
print("Generated CSV files:")
for label, filename in output_files.items():
    print(f"  {label} version: {filename}")


Creating 1MB version -> reviews_1MB.csv (target size: 1048576 bytes)
File reviews_1MB.csv created, size = 1050596 bytes

Creating 10MB version -> reviews_10MB.csv (target size: 10485760 bytes)
File reviews_10MB.csv created, size = 10486164 bytes

Creating 100MB version -> reviews_100MB.csv (target size: 104857600 bytes)
File reviews_100MB.csv created, size = 104863073 bytes

Creating 1GB version -> reviews_1GB.csv (target size: 1073741824 bytes)
File reviews_1GB.csv created, size = 1073742715 bytes

Generated CSV files:
  1MB version: reviews_1MB.csv
  10MB version: reviews_10MB.csv
  100MB version: reviews_100MB.csv
  1GB version: reviews_1GB.csv


### **Dataset preview before pre-processing :**

In [6]:
import pandas as pd
from tabulate import tabulate

# Load the dataset (the 1MB version here)
df = pd.read_csv("reviews_1MB.csv")

# Display the first 5 rows of the complete dataset
print("Dataset preview before removing unnecessary columns:")
print(tabulate(df.head(5), headers='keys', tablefmt='fancy_grid', showindex=False))


Dataset preview before removing unnecessary columns:
╒═════════════════════╤══════════════════════════════════════════════════════════════════════╤═════════════════╤═════════════════╤══════════════════════════════╤══════════════════════╤════════════════╤═══════════════╤════════════════════════════════════════╤════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╕
│ product/productId   │ product

## **Filtering the reviews : keeping the text and the review score**

In the context of NLP, particularly when working with models like BERT-base, we only need the textual data (i.e., the review text) and, if applicable, a label (here, the review score) for training purposes. By keeping only these relevant columns and renaming them for clarity, we streamline the dataset to focus on the input and output required for our NLP tasks, reducing unnecessary overhead during preprocessing and model training.

In [7]:
import pandas as pd

# Define file names for each dataset version
file_versions = {
    "1MB": "reviews_1MB.csv",
    "10MB": "reviews_10MB.csv",
    "100MB": "reviews_100MB.csv",
    "1GB": "reviews_1GB.csv"
}

# Define the columns to keep for BERT training
columns_to_keep = ["review/text", "review/score"]

# Dictionary to store the cleaned DataFrames
datasets = {}

# Create DataFrames for each dataset version
for version, filename in file_versions.items():
    # Load the dataset from CSV
    df = pd.read_csv(filename)


    # Keep only the required columns and create a copy
    df = df[columns_to_keep].copy()

    # Rename columns for easier handling
    df.rename(columns={"review/text": "text", "review/score": "score"}, inplace=True)

    # Store the cleaned DataFrame in the dictionary
    datasets[version] = df

    # Print confirmation info
    print(f"Dataset {version} loaded from {filename}.")
    print(f"  New shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}\n")

Dataset 1MB loaded from reviews_1MB.csv.
  New shape: (1338, 2)
  Columns: ['text', 'score']

Dataset 10MB loaded from reviews_10MB.csv.
  New shape: (12585, 2)
  Columns: ['text', 'score']

Dataset 100MB loaded from reviews_100MB.csv.
  New shape: (122252, 2)
  Columns: ['text', 'score']

Dataset 1GB loaded from reviews_1GB.csv.
  New shape: (1225518, 2)
  Columns: ['text', 'score']



### Pandas df we will be working for the prediction :

In [8]:
df_1MB = datasets["1MB"]
df_10MB = datasets["10MB"]
df_100MB = datasets["100MB"]
df_1GB = datasets["1GB"]

df_1GB.head(100)

,text,score
0,These pants are great for putting on over shor...,5.0
1,Dominik is a mediterranean guy who loves sex. ...,2.0
2,Length:: 4:47 Mins,4.0
3,This author actually believes that she can tel...,2.0
4,This book is okay. I've actually written 2 scr...,2.0
...,...,...
95,I have this in my bedroom and I drift off to s...,5.0
96,I bought this to replace a broken Brookstone s...,1.0
97,Absolute garbage. Pump is too loud sound is ve...,1.0
98,"Great product, arrived quickly and works prope...",4.0


In [9]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(torch.cuda.is_available())  # Should return True
# print(torch.cuda.get_device_name(0))  # Should print the GPU name
print(device)

cuda


In [10]:
import re

def clean_text(text):
    text = re.sub(r"\s+", " ", str(text))  # Normalize spaces
    text = re.sub(r"https?://\S+|www\.\S+", "", text)  # Remove URLs
    text = re.sub(r"[^a-zA-Z0-9.,!?\"' ]", "", text)  # Remove unwanted characters
    text = text.lower()  # Convert to lowercase
    text = text.strip()  # Trim leading/trailing spaces
    return text

df_1MB["text"] = df_1MB["text"].apply(clean_text)
df_10MB["text"] = df_10MB["text"].apply(clean_text)
df_100MB["text"] = df_100MB["text"].apply(clean_text)
df_1GB["text"] = df_1GB["text"].apply(clean_text)
df_100MB.head(100)

,text,score
0,these stories are pervaded with a certain sadn...,5.0
1,"these stories about new york, even when read f...",5.0
2,i wasn't sure about this book but decided to g...,5.0
3,flannery o'connor is responsible for my purcha...,5.0
4,this book was interesting. i feel not everyone...,4.0
...,...,...
95,i heard this album just about the same time i ...,4.0
96,i just want to say that this record blows my m...,5.0
97,fun9 is a treat because it features some of ta...,4.0
98,i was drawn to takako minekawa via ryuichi sak...,4.0


# BERT Tokenization
Loads the BERT tokenizer specifically for the uncased (lowercase) version
This matches the vocabulary BERT was originally trained on

In [11]:
from transformers import BertTokenizer
# https://huggingface.co/docs/transformers/en/main_classes/tokenizer

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenization function
def tokenize_text(text):
    return tokenizer(text, padding="max_length", truncation=True, max_length=512, return_tensors="pt") # Need to revisit params

# Apply tokenizer to the 'text' column
df_1MB["tokens"] = df_1MB["text"].apply(lambda x: tokenize_text(x)["input_ids"].squeeze().tolist())
# df_10MB["tokens"] = df_10MB["text"].apply(lambda x: tokenize_text(x)["input_ids"].squeeze().tolist())
# df_100MB["tokens"] = df_100MB["text"].apply(lambda x: tokenize_text(x)["input_ids"].squeeze().tolist())
# df_1GB["tokens"] = df_1GB["text"].apply(lambda x: tokenize_text(x)["input_ids"].squeeze().tolist())
# df_1MB.head(100)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# Sentiment Labelling
*  1-2 stars → 0 (Negative)
*  3 stars → 1 (Neutral)
*  4-5 stars → 2 (Positive)







In [12]:
# Define function to categorize sentiment
def encode_sentiment(score):
    if score <= 2:
        return 0  # Negative
    elif score == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

# Apply function to the 'score' column
df_1MB["label"] = df_1MB["score"].apply(encode_sentiment)
# df_10MB["label"] = df_10MB["score"].apply(encode_sentiment)
# df_100MB["label"] = df_100MB["score"].apply(encode_sentiment)
# df_1GB["label"] = df_1GB["score"].apply(encode_sentiment)
df_1MB.head(100)



,text,score,tokens,label
0,i own the austin reed dartmouth blazer in ever...,4.0,"[101, 1045, 2219, 1996, 5899, 7305, 16960, 153...",2
1,got these last christmas as a gag gift. they a...,5.0,"[101, 2288, 2122, 2197, 4234, 2004, 1037, 1820...",2
2,gave this to my dad for a gag gift after direc...,3.0,"[101, 2435, 2023, 2000, 2026, 3611, 2005, 1037...",1
3,this is only for julie strain fans. it's a col...,4.0,"[101, 2023, 2003, 2069, 2005, 7628, 10178, 459...",2
4,i hope a lot of people hear this cd. we need m...,5.0,"[101, 1045, 3246, 1037, 2843, 1997, 2111, 2963...",2
...,...,...,...,...
95,"this was a great book,i just could not put it ...",4.0,"[101, 2023, 2001, 1037, 2307, 2338, 1010, 1045...",2
96,this is a wonderful book that will keep you gu...,5.0,"[101, 2023, 2003, 1037, 6919, 2338, 2008, 2097...",2
97,just as predicted the first chapter of the boo...,4.0,"[101, 2074, 2004, 10173, 1996, 2034, 3127, 199...",2
98,"i thought this book was brilliant, but yet rea...",5.0,"[101, 1045, 2245, 2023, 2338, 2001, 8235, 1010...",2


# Data Splitting
1. Splits data into:
*  Training set (70%)
*  Validation set (15%)
*  Test set (15%)
2. Uses stratified sampling to maintain class distribution

In [13]:
from sklearn.model_selection import train_test_split

# Split into training (70%), validation (15%), and test (15%)
train_df, temp_df = train_test_split(df_1MB, test_size=0.3, random_state=42, stratify=df_1MB["label"]) # Do we want to stratify with a different class?
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

train_df.size, val_df.size, test_df.size

(3744, 804, 804)

# BERT Model Setup

1. Tokenizer: Uses BERT's tokenizer to convert text to numerical tokens.
              BERT requires text to be converted into numerical tokens that it can process.
              The tokenizer handles:
              ->Splitting text into subwords/wordpieces (BERT's vocabulary)
              ->Adding special tokens like [CLS] and [SEP]
              ->Padding/truncating to a fixed length (512 tokens for BERT)
              ->Creating attention masks to distinguish real tokens from padding

              Key Parameters:
              * padding="max_length": Ensures all sequences are padded to 512 tokens
              * truncation=True: Cuts off text exceeding 512 tokens
              * return_tensors="pt": Returns PyTorch tensors instead of Python lists

   Creating PyTorch Datasets:
              ->To efficiently feed data to the model during training
              ->Handles:
                        Tokenization on-the-fly
                        Batching
                        Shuffling
             -> Output Format:
                * input_ids: Numerical token IDs (shape: [batch_size, 512])
                * attention_mask: Binary mask (1 for real tokens, 0 for padding) (shape: [batch_size, 512])
                * labels: Sentiment class (0, 1, or 2)

2. BERT Model Initialization: Loads pretrained BERT-base model with:
*  3 output classes (negative, neutral, positive)
*  Sequence classification head
*  Adapts BERT for sentiment analysis (3-class classification)
    
        -> Key Components
          * bert-base-uncased: Pre-trained BERT model (lowercase text)
          * Classification Head: A linear layer on top of BERT's [CLS] token output
          * num_labels=3: Configures the model for 3 output classes

3. Optimizer: Uses AdamW with learning rate 2e-5
  * AdamW ("Adam with Weight Decay") is the default optimizer for fine-tuning  BERT (and most transformer models). AdamW decouples weight decay from gradient updates, following the original L2 regularization intent.
  * BERT has 110M+ parameters, so proper regularization is crucial to avoid overfitting.
  * Adjusts learning rates per parameter (e.g., lower for embeddings, higher for classifier head).
  * Handles sparse gradients (common in NLP due to rare words).


In [15]:
import torch
from transformers import BertForSequenceClassification, BertConfig
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert DataFrames to PyTorch datasets
def create_dataset(df, tokenizer, max_length=512, size_limit_mb=1):
    input_ids = []
    attention_masks = []

    # Estimate rows for 1 MB, assuming ~1.2 KB per row (text + label)
    max_rows = (size_limit_mb * 1024 * 1024) // 1200
    df = df.sample(n=min(max_rows, len(df)))
    for text in df['text']:
        encoded_dict = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            padding="max_length", # Changed from pad_to_max_length
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])

    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(df['label'].values)

    return TensorDataset(input_ids, attention_masks, labels)

# Create datasets
train_dataset = create_dataset(train_df, tokenizer)
val_dataset = create_dataset(val_df, tokenizer)
test_dataset = create_dataset(test_df, tokenizer)

# Create data loaders
batch_size = 16
train_dataloader = DataLoader(
    train_dataset,
    sampler=RandomSampler(train_dataset),
    batch_size=batch_size
)
val_dataloader = DataLoader(
    val_dataset,
    sampler=SequentialSampler(val_dataset),
    batch_size=batch_size
)
test_dataloader = DataLoader(
    test_dataset,
    sampler=SequentialSampler(test_dataset),
    batch_size=batch_size
)

# Load BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3, # Negative, Neutral, Positive
    output_attentions=False,
    output_hidden_states=False
)
model.to(device)

# Set up optimizer
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Training Process - Fine-tunes BERT on the Amazon reviews dataset

*   Processes data in batches (size 16) - Avoids memory constraints
*   Tracks training and validation loss/accuracy
*   Runs for 4 epochs





In [16]:
def train_model(model, train_dataloader, val_dataloader, epochs=4):
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        print("-" * 10)

        # Training
        model.train()
        total_train_loss = 0

        for batch in train_dataloader:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            model.zero_grad()

            outputs = model(
                b_input_ids,
                attention_mask=b_input_mask,
                labels=b_labels
            )

            loss = outputs.loss
            total_train_loss += loss.item()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        avg_train_loss = total_train_loss / len(train_dataloader)
        print(f"Average training loss: {avg_train_loss:.2f}")

        # Validation
        model.eval()
        total_eval_accuracy = 0
        total_eval_loss = 0

        for batch in val_dataloader:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            with torch.no_grad():
                outputs = model(
                    b_input_ids,
                    attention_mask=b_input_mask,
                    labels=b_labels
                )

            loss = outputs.loss
            logits = outputs.logits

            total_eval_loss += loss.item()

            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.to('cpu').numpy()

            total_eval_accuracy += accuracy_score(label_ids, np.argmax(logits, axis=1))

        avg_val_accuracy = total_eval_accuracy / len(val_dataloader)
        print(f"Validation Accuracy: {avg_val_accuracy:.2f}")
        avg_val_loss = total_eval_loss / len(val_dataloader)
        print(f"Validation Loss: {avg_val_loss:.2f}")

    print("Training complete!")

# Evaluation


*  Calculates precision, recall, and F1-score for each class
*  Reports overall accuracy (79.38% on test set)



In [17]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions, true_labels = [], []

    for batch in test_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model(
                b_input_ids,
                attention_mask=b_input_mask
            )

        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        predictions.append(logits)
        true_labels.append(label_ids)

    # Combine results for all batches
    flat_predictions = np.concatenate(predictions, axis=0)
    flat_predictions = np.argmax(flat_predictions, axis=1).flatten()
    flat_true_labels = np.concatenate(true_labels, axis=0)

    # Print classification report
    print(classification_report(
        flat_true_labels,
        flat_predictions,
        target_names=['Negative', 'Neutral', 'Positive']
    ))

    # Calculate accuracy
    accuracy = accuracy_score(flat_true_labels, flat_predictions)
    print(f"Test Accuracy: {accuracy:.4f}")

In [18]:
# Train the model
train_model(model, train_dataloader, val_dataloader, epochs=4)

# Evaluate on test set
evaluate_model(model, test_dataloader)

Epoch 1/4
----------
Average training loss: 0.63
Validation Accuracy: 0.82
Validation Loss: 0.51
Epoch 2/4
----------
Average training loss: 0.39
Validation Accuracy: 0.88
Validation Loss: 0.33
Epoch 3/4
----------
Average training loss: 0.24
Validation Accuracy: 0.90
Validation Loss: 0.30
Epoch 4/4
----------
Average training loss: 0.13
Validation Accuracy: 0.90
Validation Loss: 0.32
Training complete!
              precision    recall  f1-score   support

    Negative       0.72      0.54      0.62        24
     Neutral       0.40      0.46      0.43        13
    Positive       0.93      0.95      0.94       164

    accuracy                           0.87       201
   macro avg       0.68      0.65      0.66       201
weighted avg       0.87      0.87      0.87       201

Test Accuracy: 0.8706
